# 06 — East Model v4 — Retrain with TPI
**Project IceWave | Notebook 06**

Adds `tpi_15m` as a 6th feature to the east Random Forest model.

## Why TPI matters
The v3 east model scored E02 (TPI=-5m, valley floor) and E32 (TPI=+40m, ridgetop) identically
at composite=1.000 — it couldn't tell them apart. TPI from LiDAR notebook 05 fixes this.

## What this notebook does
1. Load 40 east training points + their 15m TPI values from USGS 3DEP
2. Fetch TPI for background points too (same TNM endpoint)
3. Retrain east RF with features: elev, slope, aspect, TRI, lith_score, **tpi_15m**
4. Compare v4 AUC vs v3 AUC 0.846
5. Re-score all 50 targets with v4 model
6. Save updated model and target CSV

In [ ]:
# ── Cell 1: Imports and config ─────────────────────────────────────────────
import requests
import numpy as np
import pandas as pd
import joblib
import rasterio
from scipy.ndimage import uniform_filter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

DATA_DIR   = Path('../data')
LIDAR_DIR  = DATA_DIR / 'lidar'
MODEL_DIR  = DATA_DIR / 'model'
OUT_DIR    = Path('../outputs')
LIDAR_DIR.mkdir(parents=True, exist_ok=True)

TNM_URL    = 'https://elevation.nationalmap.gov/arcgis/rest/services/3DEPElevation/ImageServer'
TPI_RADIUS = 100   # pixels @ 15m = 1500m neighborhood (same as nb05)
MAX_PX     = 500
BUFFER_DEG = 0.045
FETCH_RES  = 0.00016
CASCADE    = -121.5

print('Imports OK')

In [ ]:
# ── Cell 2: Load training data ─────────────────────────────────────────────
# Load v3 training CSV — 40 presence points + background
train_csv = DATA_DIR / 'pbdb' / 'icewave_east_expanded.csv'
df_train  = pd.read_csv(train_csv)

# Load lidar CSV for TPI values at target locations (already fetched in nb05)
lidar_csv = MODEL_DIR / 'icewave_v3_top50_lidar.csv'
df_lidar  = pd.read_csv(lidar_csv, index_col='rank')

# Load v3 model to access background points
v3_model  = joblib.load(MODEL_DIR / 'icewave_rf_east_v3.joblib')

print(f'Training CSV: {len(df_train)} rows, columns: {list(df_train.columns)}')
print(f'LiDAR CSV:    {len(df_lidar)} rows')
print(f'V3 model features: {v3_model.n_features_in_}')

In [ ]:
# ── Cell 3: TPI fetch function (same as nb05) ──────────────────────────────

def fetch_tpi_at_point(lat, lon, label='pt'):
    """
    Fetch a small DTM tile from TNM and return TPI at the center point.
    Uses a small buffer (0.01deg ~1km) for speed — sufficient for single-point TPI.
    """
    buf = 0.012   # small tile for speed
    xmin, xmax = lon - buf, lon + buf
    ymin, ymax = lat - buf, lat + buf

    # TPI at 1500m neighborhood needs ~100px radius
    # At 0.00016deg/px and 0.024deg wide = 150px — fine
    ncols = nrows = min(MAX_PX, int(buf * 2 / FETCH_RES))

    params = {
        'bbox':      f'{xmin},{ymin},{xmax},{ymax}',
        'bboxSR':    '4326',
        'size':      f'{ncols},{nrows}',
        'imageSR':   '4326',
        'format':    'tiff',
        'pixelType': 'F32',
        'noDataInterpretation': 'esriNoDataMatchAny',
        'interpolation': 'RSP_BilinearInterpolation',
        'f': 'image',
    }
    r = requests.get(f'{TNM_URL}/exportImage', params=params, timeout=30)
    r.raise_for_status()
    if len(r.content) < 1000:
        return np.nan

    with rasterio.MemoryFile(r.content) as mf:
        with mf.open() as src:
            dem = src.read(1).astype(np.float32)
            tf  = src.transform
            nd  = src.nodata

    if nd is not None:
        dem[dem == nd] = np.nan
    dem[dem < -500] = np.nan

    # TPI: center pixel vs neighborhood mean
    dem_f = dem.copy()
    nan_mask = np.isnan(dem_f)
    if nan_mask.any():
        tmp = uniform_filter(np.where(nan_mask, 0.0, dem_f), size=5)
        cnt = uniform_filter((~nan_mask).astype(float), size=5)
        dem_f[nan_mask] = np.where(cnt[nan_mask] > 0, tmp[nan_mask] / cnt[nan_mask], 0)

    nbr_mean = uniform_filter(dem_f, size=TPI_RADIUS * 2 + 1)
    tpi      = dem_f - nbr_mean

    # Extract center pixel value
    c = int((lon - tf.c) / tf.a)
    r_idx = int((lat - tf.f) / tf.e)
    if 0 <= r_idx < tpi.shape[0] and 0 <= c < tpi.shape[1]:
        return float(tpi[r_idx, c])
    return np.nan


# Test
print('Testing TPI fetch at E02 (46.3003N, 120.1336W)...')
tpi_test = fetch_tpi_at_point(46.3003, -120.1336)
print(f'TPI = {tpi_test:.2f}m  (expected ~-5.4m)')

In [ ]:
# ── Cell 4: Fetch TPI for all training points ──────────────────────────────
# Training CSV has presence + background points
# Check column names and identify lat/lon/label columns

print('Training data columns:', df_train.columns.tolist())
print(df_train.head(3))
print()

# Identify lat/lon columns (adjust if needed)
lat_col = 'lat' if 'lat' in df_train.columns else 'latitude'
lon_col = 'lon' if 'lon' in df_train.columns else 'longitude'
label_col = 'presence' if 'presence' in df_train.columns else 'label'

print(f'Using: lat={lat_col}, lon={lon_col}, label={label_col}')

In [ ]:
# ── Cell 5: Fetch TPI for each training point ──────────────────────────────
# This takes ~2-3 min depending on point count
# Caches results so re-running is fast

cache_path = DATA_DIR / 'pbdb' / 'icewave_east_tpi_cache.csv'

if cache_path.exists():
    print(f'Loading cached TPI values from {cache_path}')
    tpi_cache = pd.read_csv(cache_path, index_col=0)
    df_train['tpi_15m'] = tpi_cache['tpi_15m'].values
    print(f'Loaded {tpi_cache["tpi_15m"].notna().sum()} cached TPI values')
else:
    print(f'Fetching TPI for {len(df_train)} training points...')
    tpis = []
    for i, row in df_train.iterrows():
        try:
            tpi = fetch_tpi_at_point(row[lat_col], row[lon_col])
            tpis.append(tpi)
            if i % 20 == 0:
                print(f'  {i+1}/{len(df_train)}  last TPI={tpi:.2f}' if not np.isnan(tpi) else f'  {i+1}/{len(df_train)}  NaN')
        except Exception as e:
            tpis.append(np.nan)
            print(f'  {i}: failed ({e})')

    df_train['tpi_15m'] = tpis
    # Cache
    pd.DataFrame({'tpi_15m': tpis}).to_csv(cache_path)
    print(f'Saved cache to {cache_path}')

good = df_train['tpi_15m'].notna().sum()
print(f'\nTPI valid: {good}/{len(df_train)} points')
print(f'Presence TPI range: {df_train[df_train[label_col]==1]["tpi_15m"].describe()}')
print(f'Background TPI range: {df_train[df_train[label_col]==0]["tpi_15m"].describe()}')

In [ ]:
# ── Cell 6: TPI distribution — presence vs background ─────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor='#0d1f0d')

pres = df_train[df_train[label_col]==1]['tpi_15m'].dropna()
back = df_train[df_train[label_col]==0]['tpi_15m'].dropna()

vmax = max(abs(pres).quantile(0.95), abs(back).quantile(0.95), 20)
bins = np.linspace(-vmax, vmax, 40)

ax = axes[0]
ax.hist(pres, bins=bins, alpha=0.7, color='#f4a261', label=f'Presence (n={len(pres)})')
ax.hist(back, bins=bins, alpha=0.5, color='#00b4d8', label=f'Background (n={len(back)})')
ax.axvline(0, color='#c9a84c', lw=1, ls='--')
ax.set_xlabel('TPI (m)', color='#aaaaaa')
ax.set_ylabel('Count', color='#aaaaaa')
ax.set_title('TPI Distribution — Presence vs Background', color='#c9a84c', fontweight='bold')
ax.legend()
ax.set_facecolor('#1a3a1a')
ax.tick_params(colors='#aaaaaa')

ax2 = axes[1]
ax2.boxplot([pres, back], labels=['Presence', 'Background'],
            patch_artist=True,
            boxprops=dict(facecolor='#1a3a1a', color='#c9a84c'),
            medianprops=dict(color='#f4a261', lw=2),
            whiskerprops=dict(color='#aaaaaa'),
            capprops=dict(color='#aaaaaa'),
            flierprops=dict(marker='o', color='#aaaaaa', ms=3))
ax2.axhline(0, color='#c9a84c', lw=1, ls='--')
ax2.set_ylabel('TPI (m)', color='#aaaaaa')
ax2.set_title('TPI Boxplot', color='#c9a84c', fontweight='bold')
ax2.set_facecolor('#1a3a1a')
ax2.tick_params(colors='#aaaaaa')

for ax in axes:
    for sp in ax.spines.values():
        sp.set_edgecolor('#3a5a3a')

fig.suptitle('TPI as discriminator: presence sites should skew negative (valley floors)',
             color='#c9a84c', fontsize=10)
fig.patch.set_facecolor('#0d1f0d')
plt.tight_layout()
plt.savefig(OUT_DIR / 'tpi_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1f0d')
plt.show()

In [ ]:
# ── Cell 7: Build feature matrix ───────────────────────────────────────────

# V3 features
FEATURES_V3 = ['elevation', 'slope', 'aspect', 'tri', 'lith_score']
# V4 adds TPI
FEATURES_V4 = ['elevation', 'slope', 'aspect', 'tri', 'lith_score', 'tpi_15m']

# Drop rows missing TPI (fill with 0 as fallback for background points with failed fetch)
df_model = df_train.copy()
n_missing = df_model['tpi_15m'].isna().sum()
if n_missing > 0:
    print(f'Warning: {n_missing} points missing TPI — filling with 0 (flat plain assumption)')
    df_model['tpi_15m'] = df_model['tpi_15m'].fillna(0)

X_v3 = df_model[FEATURES_V3].values
X_v4 = df_model[FEATURES_V4].values
y    = df_model[label_col].values

print(f'X_v3: {X_v3.shape}  X_v4: {X_v4.shape}  y: {y.shape}')
print(f'Presence: {y.sum()}  Background: {(y==0).sum()}')

In [ ]:
# ── Cell 8: Cross-validated AUC comparison ────────────────────────────────

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_params = dict(n_estimators=500, max_depth=6, min_samples_leaf=3,
                 class_weight='balanced', random_state=42, n_jobs=-1)

rf_v3 = RandomForestClassifier(**rf_params)
rf_v4 = RandomForestClassifier(**rf_params)

print('Cross-validating v3 (5 features)...')
auc_v3 = cross_val_score(rf_v3, X_v3, y, cv=cv, scoring='roc_auc')
print(f'  V3 AUC: {auc_v3.mean():.3f} ± {auc_v3.std():.3f}  (published: 0.846 ± 0.073)')

print('Cross-validating v4 (6 features, +TPI)...')
auc_v4 = cross_val_score(rf_v4, X_v4, y, cv=cv, scoring='roc_auc')
print(f'  V4 AUC: {auc_v4.mean():.3f} ± {auc_v4.std():.3f}')

delta = auc_v4.mean() - auc_v3.mean()
print(f'\n  Delta: {delta:+.3f}')
if delta > 0.01:
    print('  ✓ TPI improves the model')
elif delta > -0.01:
    print('  ~ TPI is neutral — model unchanged')
else:
    print('  ✗ TPI hurts — check TPI fetch quality')

In [ ]:
# ── Cell 9: Fit final v4 model and feature importance ─────────────────────

rf_v4_final = RandomForestClassifier(**rf_params)
rf_v4_final.fit(X_v4, y)

# Feature importance
imp = rf_v4_final.feature_importances_
fig, ax = plt.subplots(figsize=(8, 4), facecolor='#0d1f0d')
colors = ['#00b4d8' if f != 'tpi_15m' else '#7b68ee' for f in FEATURES_V4]
bars = ax.barh(FEATURES_V4, imp, color=colors)
ax.set_xlabel('Feature Importance', color='#aaaaaa')
ax.set_title('IceWave East v4 — Feature Importance (+TPI in purple)',
             color='#c9a84c', fontweight='bold')
ax.set_facecolor('#1a3a1a')
ax.tick_params(colors='#aaaaaa')
for sp in ax.spines.values():
    sp.set_edgecolor('#3a5a3a')
for bar, val in zip(bars, imp):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left', color='#aaaaaa', fontsize=8)
fig.patch.set_facecolor('#0d1f0d')
plt.tight_layout()
plt.savefig(OUT_DIR / 'feature_importance_v4.png', dpi=150, bbox_inches='tight', facecolor='#0d1f0d')
plt.show()

print('Feature importances:')
for f, i in sorted(zip(FEATURES_V4, imp), key=lambda x: -x[1]):
    bar = '█' * int(i * 100)
    print(f'  {f:12s} {i:.3f}  {bar}')

In [ ]:
# ── Cell 10: Re-score all east targets with v4 model ──────────────────────

targets_csv = MODEL_DIR / 'icewave_v3_top50.csv'
df_targets  = pd.read_csv(targets_csv, index_col='rank')
east = df_targets[df_targets['ecoregion'] == 'east'].copy()

# TPI already in lidar CSV
df_lidar_loaded = pd.read_csv(MODEL_DIR / 'icewave_v3_top50_lidar.csv', index_col='rank')
east['tpi_15m'] = df_lidar_loaded['tpi_15m']
east['tpi_15m'] = east['tpi_15m'].fillna(0)  # fill missing with flat

X_east_v4 = east[FEATURES_V4].values
probs_v4   = rf_v4_final.predict_proba(X_east_v4)[:, 1]
east['prob_v4']  = probs_v4
east['comp_v4']  = 0.8 * probs_v4 + 0.2 * east['lith_score']
# Normalize
cmax = east['comp_v4'].max()
east['comp_v4_norm'] = east['comp_v4'] / cmax if cmax > 0 else east['comp_v4']

print('East targets — v3 vs v4 score comparison:')
print(f'{"Rank":6s} {"v3":6s} {"v4":6s} {"delta":7s} {"TPI":8s} {"Class"}')
print('-' * 65)
for rank, row in east.iterrows():
    v3 = row.get('composite_norm', 0)
    v4 = row['comp_v4_norm']
    d  = v4 - v3
    tpi = row['tpi_15m']
    tpi_class = ('▼VF' if tpi < -5 else '▽LT' if tpi < -1 else '─FP' if tpi < 1 else '△LR' if tpi < 5 else '▲RT')
    arrow = '↑' if d > 0.02 else ('↓' if d < -0.02 else '=')
    print(f'E{int(rank):02d}    {v3:.3f}  {v4:.3f}  {d:+.3f} {arrow}  {tpi:+6.1f}m  {tpi_class}')

In [ ]:
# ── Cell 11: Save v4 model and updated targets ────────────────────────────

# Save model
v4_model_path = MODEL_DIR / 'icewave_rf_east_v4.joblib'
joblib.dump(rf_v4_final, v4_model_path)
print(f'Saved model: {v4_model_path}')

# Update targets CSV
df_out = df_targets.copy()
for rank, row in east.iterrows():
    df_out.loc[rank, 'prob_v4']      = round(row['prob_v4'], 4)
    df_out.loc[rank, 'comp_v4_norm'] = round(row['comp_v4_norm'], 4)

out_csv = MODEL_DIR / 'icewave_v4_top50.csv'
df_out.to_csv(out_csv)
print(f'Saved targets: {out_csv}')

# Summary
print(f'''
{'='*55}
  IceWave East Model — v3 vs v4 Summary
{'='*55}
  V3 AUC: {auc_v3.mean():.3f} ± {auc_v3.std():.3f}  (5 features)
  V4 AUC: {auc_v4.mean():.3f} ± {auc_v4.std():.3f}  (6 features, +TPI)
  Delta:  {delta:+.3f}

  V4 re-ranks:
  - Valley floor targets (TPI < -5m) should score higher
  - Ridge targets (TPI > +1m) should score lower
  - E32 (TPI +40m) should drop significantly
{'='*55}''')